# QEC QCNN decoder — Kaggle GPU training

Trains the two QCNN decoders (Cong + Hybrid QCCNN) on a GPU using PennyLane's
Torch-native `default.qubit` statevector, which runs on CUDA and backprops the
whole minibatch at once.

**Before running:**
1. Notebook settings -> **Accelerator: GPU** (T4 or P100).
2. Notebook settings -> **Internet: On** (needed to clone the repo + pip install).

CPU tests and the HPC path are unaffected — this only sets
`QEC_QML_DEVICE=default.qubit` and `--device cuda`.

In [ ]:
# 1. Clone the PR branch (public repo)
%cd /kaggle/working
!rm -rf Quantum-Error-Correction-Decoder
!git clone -b feat/qec-qcnn-decoder --depth 1 \
    https://github.com/TuanKiet16/Quantum-Error-Correction-Decoder.git
%cd Quantum-Error-Correction-Decoder

In [ ]:
# 2. Install. Torch is preinstalled on Kaggle GPU images; install the rest.
!pip install -q stim pymatching pennylane 'pennylane-lightning' loguru
# The package itself (pulls nothing heavy since torch already present):
!pip install -q -e . --no-deps

In [ ]:
# 3. Confirm the GPU is visible
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Enable GPU accelerator in notebook settings'

In [ ]:
# 4. GPU backend for PennyLane, for the whole process
import os
os.environ['QEC_QML_DEVICE'] = 'default.qubit'

In [ ]:
# 5. Quick smoke run to prove the GPU path works (seconds)
!QEC_QML_DEVICE=default.qubit python -m qec_decoder.train \
    --model qcnn_cong --d 3 --ps 0.005 --shots 400 --epochs 1 \
    --batch-size 128 --device cuda

In [ ]:
# 6. Full training. Edit MODELS / DISTANCES / PS / SHOTS / EPOCHS as budget allows.
#    Kaggle GPU sessions cap at ~9h; start small (d=3) and grow.
import subprocess, itertools

MODELS    = ['qcnn_cong', 'qcnn_hybrid']
DISTANCES = [3, 5]
PS        = ['0.003', '0.005', '0.008', '0.01', '0.015']
SHOTS     = 20000
EPOCHS    = 30

for model, d in itertools.product(MODELS, DISTANCES):
    cmd = ['python', '-m', 'qec_decoder.train',
           '--model', model, '--d', str(d), '--ps', *PS,
           '--shots', str(SHOTS), '--epochs', str(EPOCHS),
           '--batch-size', '256', '--device', 'cuda']
    print('>>>', ' '.join(cmd))
    env = {**os.environ, 'QEC_QML_DEVICE': 'default.qubit'}
    subprocess.run(cmd, env=env, check=True)

In [ ]:
# 7. Bundle checkpoints + run records to download from the Output tab
!mkdir -p /kaggle/working/out
!cp -r checkpoints results /kaggle/working/out/ 2>/dev/null || true
!cd /kaggle/working && zip -qr qec_qcnn_output.zip out && echo 'wrote qec_qcnn_output.zip'
!ls -la /kaggle/working/checkpoints /kaggle/working/Quantum-Error-Correction-Decoder/checkpoints 2>/dev/null || true